In [ ]:
import pandas as pd
import numpy as np
import os
import time
from Python_Modules.Train_HF_Models_Original import create_classification_dataset, train_hf_classification_model

import random
import numpy as np
import torch
import transformers

def set_seed(seed=1):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if using multi-GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    transformers.set_seed(seed)
set_seed(1)

In [ ]:
HF_MODEL_DICT = {'ViT': 'google/vit-base-patch16-224-in21k', 'ConvNeXtBase': 'facebook/convnext-base-384-22k-1k', 'ResNet': 'microsoft/resnet-152'}

## Run results

In [ ]:
CSV_PATH = 'Results.csv'
df = pd.read_csv(CSV_PATH)
df.tail(2)

In [ ]:
Models = ['ConvNeXtBase', 'ViT', 'ResNet']
datasets = [
 'AIDA',
 'Brand-Styles',
 'BrandSelfie',
 'E-Commerce',
 'Emotion-6',
 'FILGRIM',
 'FashionImages',
 'GenImgNet',
 'Generated-Ethnicity',
 'Generated-Gender',
 'KonIQ',
 'LogoDet-3K',
 'StoreItemColor',
 'UnBiasedEmo',
 'Unsplash-Images',
 'good-bad',
 'intel-sceneries',
 'sentiment']

In [ ]:
EPOCHS = 32
BATCH_SIZES = [16,32,64]
LEARNING_RATES = [0.01, 0.001, 1e-5]
MODEL_OUT_DIR = 'HF_results_Benchmark'

In [ ]:
for MODEL_NAME in Models: 
    for N_Samples in [200, 1000]:
        for DATASET_NAME in datasets: 
            for LEARNING_RATE in LEARNING_RATES:
                for BATCH_SIZE in BATCH_SIZES:
                    if (DATASET_NAME == 'AIDA') & (N_Samples == 1000):
                        continue

                    df = pd.read_csv(CSV_PATH)
                    if df.loc[((df.Model == MODEL_NAME) & (df.Dataset == DATASET_NAME) & (df.N_Training_Samples == N_Samples) & (df.BATCH_SIZE == BATCH_SIZE) & (df.LEARNING_RATE == LEARNING_RATE))].shape[0] == 0:
                        try:
                            print(MODEL_NAME, ": ", DATASET_NAME, BATCH_SIZE, LEARNING_RATE)
                            HF_MODEL_NAME = HF_MODEL_DICT[MODEL_NAME]

                            DATASET_CSV = pd.read_csv(f"CSV_Training_{N_Samples}/{DATASET_NAME}.csv")
                            DATASET_CSV_HOLDOUT = pd.read_csv(f"CSV_Holdout/{DATASET_NAME}.csv")

                            train_labels = set(DATASET_CSV['label']).intersection(set(DATASET_CSV_HOLDOUT['label']))
                            DATASET_CSV = DATASET_CSV[DATASET_CSV['label'].isin(train_labels)]
                            DATASET_CSV_HOLDOUT = DATASET_CSV_HOLDOUT[DATASET_CSV_HOLDOUT['label'].isin(train_labels)]
                            label_to_int = {label: str(idx) for idx, label in enumerate(sorted(train_labels))}
                            DATASET_CSV['label_num'] = DATASET_CSV['label'].map(label_to_int)
                            DATASET_CSV_HOLDOUT['label_num'] = DATASET_CSV_HOLDOUT['label'].map(label_to_int)

                            # process dataset for huggingface and train model
                            train_dataset = create_classification_dataset(DATASET_CSV, MODELNAME=HF_MODEL_NAME).shuffle(seed=1)
                            test_dataset = create_classification_dataset(DATASET_CSV_HOLDOUT, MODELNAME=HF_MODEL_NAME).shuffle(seed=1)
        
                            start_time = time.time()
                            trainer, model, metrics, optimal_epochs_trained = train_hf_classification_model(outdir=MODEL_OUT_DIR, epochs=EPOCHS, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, train_dataset=train_dataset, test_dataset=test_dataset, MODEL_NAME=HF_MODEL_NAME)
                            training_time = np.round(time.time() - start_time, 4)
                            
                            # fill dataset
                            df = pd.read_csv(CSV_PATH)
                            i = df.shape[0]
                            df.at[i, 'Model'] = MODEL_NAME
                            df.at[i, 'Dataset'] = DATASET_NAME
                            df.at[i, 'N_Training_Samples'] = N_Samples
                            df.at[i, 'BATCH_SIZE'] = BATCH_SIZE
                            df.at[i, 'LEARNING_RATE'] = LEARNING_RATE
                            df.at[i, 'EPOCHS'] = optimal_epochs_trained
                            df.at[i, 'Time_Seconds'] = training_time
                            df.at[i, 'Accuracy'] = metrics['eval_accuracy']
                            df.to_csv(CSV_PATH, index=False)
                        except Exception as e:
                            print(str(e)[:100])

## Finetune and Save ConvNeXt for Embeds

In [ ]:
MODEL_NAME = 'ConvNeXtBase'
datasets = [
 'AIDA',
 'Brand-Styles',
 'BrandSelfie',
 'E-Commerce',
 'Emotion-6',
 'FILGRIM',
 'FashionImages',
 'GenImgNet',
 'Generated-Ethnicity',
 'Generated-Gender',
 'KonIQ',
 'LogoDet-3K',
 'StoreItemColor',
 'UnBiasedEmo',
 'Unsplash-Images',
 'good-bad',
 'intel-sceneries',
 'sentiment']

MODEL_OUT_DIR = 'HF_results_Benchmark'

In [ ]:
df = pd.read_csv('Results.csv')
df = df.loc[df.Model == MODEL_NAME]
df = df.loc[df.groupby(['Model', 'N_Training_Samples', 'Dataset'])['Accuracy'].idxmax()]
df = df.reset_index(drop=True)
df.tail()

In [ ]:
for DATASET_NAME in datasets: 
    for N_Samples in [1000, 200]:
        Model_Save_Path = f'Finetuned_Convnext_Models_ForEmbeds/{DATASET_NAME}_{N_Samples}'
        if os.path.exists(Model_Save_Path) == False:
            if (DATASET_NAME == 'AIDA') & (N_Samples == 1000):
                continue
            
            try:
                hypp_row = df.loc[(df.Model == MODEL_NAME) & (df.N_Training_Samples == N_Samples) & (df.Dataset == DATASET_NAME)].to_dict(orient='records')[0]
                BATCH_SIZE = int(hypp_row['BATCH_SIZE'])
                LEARNING_RATE = hypp_row['LEARNING_RATE']
                EPOCHS = 32
                
                print(MODEL_NAME, ": ", DATASET_NAME, BATCH_SIZE, LEARNING_RATE)
                HF_MODEL_NAME = HF_MODEL_DICT[MODEL_NAME]
            
                DATASET_CSV = pd.read_csv(f"CSV_Training_{N_Samples}/{DATASET_NAME}.csv")
                
                DATASET_CSV_HOLDOUT = pd.read_csv(f"CSV_Holdout/{DATASET_NAME}.csv")
                train_labels = set(DATASET_CSV['label'])
                DATASET_CSV_HOLDOUT = DATASET_CSV_HOLDOUT[DATASET_CSV_HOLDOUT['label'].isin(train_labels)]
                label_to_int = {label: idx for idx, label in enumerate(sorted(train_labels))}
                DATASET_CSV['label_num'] = DATASET_CSV['label'].map(label_to_int).astype(str)
                DATASET_CSV_HOLDOUT['label_num'] = DATASET_CSV_HOLDOUT['label'].map(label_to_int).astype(str)
    
                DATASET_CSV['label'] = DATASET_CSV['label_num'].astype(str)
                DATASET_CSV_HOLDOUT['label'] = DATASET_CSV_HOLDOUT['label_num'].astype(str)
    
                # process dataset for huggingface and train model
                train_dataset = create_classification_dataset(DATASET_CSV, MODELNAME=HF_MODEL_NAME).shuffle(seed=1)
                test_dataset = create_classification_dataset(DATASET_CSV_HOLDOUT, MODELNAME=HF_MODEL_NAME).shuffle(seed=1)
    
                start_time = time.time()
                trainer, model, metrics, optimal_epochs_trained = train_hf_classification_model(outdir=MODEL_OUT_DIR, epochs=EPOCHS, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, train_dataset=train_dataset, test_dataset=test_dataset, MODEL_NAME=HF_MODEL_NAME)
                training_time = np.round(time.time() - start_time, 4)
                trainer.save_model(Model_Save_Path)
            except Exception as e:
                print(str(e)[:100])